# Building a LangChain SQL Demo in Jupyter Notebook

## Step 1: Installation


In [2]:
!pip install langchain langchain-community sqlalchemy pandas ollama

     ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/2.5 MB 660.6 kB/s eta 0:00:04
     ---------------------------------------- 0.0/2.5 MB 660.6 kB/s eta 0:00:04
     - -------------------------------------- 0.1/2.5 MB 491.5 kB/s eta 0:00:06
     - -------------------------------------- 0.1/2.5 MB 590.8 kB/s eta 0:00:05
     --- ------------------------------------ 0.2/2.5 MB 888.4 kB/s eta 0:00:03
     --- ------------------------------------ 0.2/2.5 MB 958.6 kB/s eta 0:00:03
     ------ --------------------------------- 0.4/2.5 MB 1.3 MB/s eta 0:00:02
     ------- -------------------------------- 0.5/2.5 MB 1.5 MB/s eta 0:00:02
     ------------- -------------------------- 0.8/2.5 MB 1.9 MB/s eta 0:00:01
     ---------------- ----------------------- 1.0/2.5 MB 2.2 MB/s eta 0:00:01
     ------------------ --------------------- 1.2/2.5 MB 2.3 MB/s eta 0:00:01
     ------------------------- -------------- 1.6/2.5 MB 3.


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Step 2: Import Libraries and Set API Key


In [ ]:
import sqlite3
import pandas as pd
from sqlalchemy import create_engine
from langchain.utilities import SQLDatabase
from langchain.llms import Ollama
from langchain_experimental.sql import SQLDatabaseChain
from langchain.agents import create_sql_agent
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
import ollama  
from langchain.chains import create_sql_query_chain
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Step 3: Create Sample Database and Data

In [ ]:
# Create a SQLite database in memory (you can change to a file path if you want to persist)
conn = sqlite3.connect('company.db')
cursor = conn.cursor()

# Create departments table
cursor.execute('''
CREATE TABLE departments (
    id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL,
    location TEXT
)
''')

# Create employees table
cursor.execute('''
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    salary REAL,
    department_id INTEGER,
    hire_date TEXT,
    FOREIGN KEY (department_id) REFERENCES departments (id)
)
''')

# Insert sample data into departments
departments_data = [
    (1, 'Engineering', 'New York'),
    (2, 'Sales', 'Chicago'),
    (3, 'Marketing', 'San Francisco'),
    (4, 'HR', 'Boston')
]
cursor.executemany('INSERT INTO departments VALUES (?, ?, ?)', departments_data)

# Insert sample data into employees
employees_data = [
    (1, 'John Doe', 75000, 1, '2020-01-15'),
    (2, 'Jane Smith', 85000, 1, '2019-03-23'),
    (3, 'Robert Johnson', 65000, 2, '2021-07-01'),
    (4, 'Emily Davis', 95000, 2, '2018-05-12'),
    (5, 'Michael Brown', 70000, 3, '2022-02-28'),
    (6, 'Sarah Wilson', 80000, 3, '2020-11-05'),
    (7, 'David Thompson', 90000, 1, '2017-09-19'),
    (8, 'Jessica Garcia', 60000, 4, '2021-04-15'),
    (9, 'Christopher Martinez', 72000, 2, '2019-08-22'),
    (10, 'Amanda Rodriguez', 88000, 1, '2018-12-10')
]
cursor.executemany('INSERT INTO employees VALUES (?, ?, ?, ?, ?)', employees_data)

# Commit changes and close connection
conn.commit()

# Let's verify our data
print("Departments:")
print(pd.read_sql_query("SELECT * FROM departments", conn))
print("\nEmployees:")
print(pd.read_sql_query("SELECT * FROM employees", conn))

# Close the connection
conn.close()

# Step 4: Set Up Database Connection for LangChain


In [43]:
# Create a SQLAlchemy engine
engine = create_engine('sqlite:///heart_disease.db')

# Create the SQLDatabase object for LangChain
db = SQLDatabase(engine)

# Let's see what tables are available
print(db.get_usable_table_names())

['Medical_Records', 'Patients', 'Treatments']


# Step 5: Initialize the Language Model


In [36]:
# Initialize the Ollama model

llm = Ollama(model="qwen2.5:7b", base_url="http://127.0.0.1:11434")
llm.invoke("Hello, Ollama!")
# Test the model with a simple prompt
response = llm.invoke("1 random word?")
print("Model test response:", response)

Model test response: Pineapple


# Step 6: Create and Use SQLDatabaseChain


In [49]:


# Step 1: Create a chain to generate SQL
query_chain = create_sql_query_chain(llm, db)

# Step 2: Generate SQL from a question
# question = "give me the list of employees who were hired after 2020 and give me their salaries and department names"
# question = "Calculate the Average salary of employees who were hired after 2020"
question = "do people with diabetes have higher blood pressure?"
question = "what is the highest blood pressure"

sql_response = query_chain.invoke({"question": question})

# Extract only the SQL query from the response
import re
match = re.search(r"SQLQuery:\s*(?:```sql\s*)?([\s\S]*?)(?:\s*```)?$", sql_response.strip(), re.DOTALL)
if match:
	clean_sql = match.group(1).strip()
else:
	raise ValueError("Could not extract SQL query from response")

# Step 3: Execute query
sql_result = db.run(clean_sql)



print("SQL Response",sql_response)
print("SQL Result:", sql_result)
# print("Analysis!!!!!!:", analysis)


SQL Response Question: What is the highest blood pressure?
SQLQuery: SELECT MAX("blood_pressure") FROM "Medical_Records";
SQL Result: [('160/100',)]


In [45]:
# Step 4: Analyze results directly
analysis_prompt = PromptTemplate.from_template("""
What does this tell about the relationship between BMI and smoking status?

{data}

Answer the user's question: {question}
""")
analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)

analysis = analysis_chain.run(data=sql_result, question=question)

In [47]:
print(analysis)

To determine if there is a correlation between BMI and smoking status based on the provided data points, we can perform a simple analysis.

The given data:
- (28.5, 1) - BMI of 28.5, Smoker (1)
- (25.2, 0) - BMI of 25.2, Non-Smoker (0)
- (30.1, 1) - BMI of 30.1, Smoker (1)
- (22.4, 0) - BMI of 22.4, Non-Smoker (0)
- (27.8, 1) - BMI of 27.8, Smoker (1)

From the data:
- There are three smokers and two non-smokers.
- Out of the five individuals, four have a BMI over 25 (considered overweight/obese), which is higher than the average BMI for a healthy adult.

Here's what we can observe:
- 3 out of 4 people with a BMI > 25 are smokers.
- 1 out of 2 non-smokers has a BMI ≤ 25, but this group is small and doesn't provide strong evidence on its own.

To better understand the correlation, we could use statistical methods such as calculating Pearson's correlation coefficient or performing a Chi-Square test. However, with only five data points, these analyses might not be very reliable due to the

In [26]:
print(sql_response)

SQLQuery: 
```sql
SELECT d.department_name
FROM departments d
JOIN (
    SELECT department_id, MAX(salary) AS max_salary
    FROM employees
    GROUP BY department_id
) e ON d.id = e.department_id
WHERE e.max_salary = (
    SELECT MIN(max_salary)
    FROM (
        SELECT MAX(salary) AS max_salary
        FROM employees
        GROUP BY department_id
    )
)
LIMIT 1;
```


# SQL Query Test

In [27]:
import sqlite3

# Connect
conn = sqlite3.connect("company.db")
cursor = conn.cursor()

# Run query
cursor.execute("""
SELECT d.department_name
FROM departments d
JOIN (
    SELECT department_id, MAX(salary) AS max_salary
    FROM employees
    GROUP BY department_id
) e ON d.id = e.department_id
WHERE e.max_salary = (
    SELECT MIN(max_salary)
    FROM (
        SELECT MAX(salary) AS max_salary
        FROM employees
        GROUP BY department_id
    )
)
LIMIT 1;
""")

# Fetch results
rows = cursor.fetchall()

# Print results
for row in rows:
    print(row)

conn.close()


('HR',)


In [ ]:
# db_chain = SQLDatabaseChain.from_llm(
#     llm=llm,
#     db=db,
#     verbose=True,
#     return_direct=True  # This returns raw SQL results instead of trying to format them
# )

# # result = db_chain.invoke("give me the list of employees who wer hired after 2020 and give me their salaries")
# result = db_chain.invoke("tell me the names of the employees who have salary above 50000 and their departments name")
# print(result)

In [8]:
# from langchain_experimental.sql import SQLDatabaseSequentialChain

# db_chain = SQLDatabaseSequentialChain.from_llm(
#     llm=llm,
#     db=db,
#     verbose=True
# )

# result = db_chain.invoke("give me the list of employees who were hired after 2020 and give me their salaries")
# print(result)


In [9]:
# from langchain.chains import create_sql_query_chain
# from langchain_experimental.sql import SQLDatabaseChain
# from langchain.prompts import PromptTemplate
# from langchain.chains import LLMChain

# # Step 1: Create a chain to generate SQL
# query_chain = create_sql_query_chain(llm, db)

# # Step 2: Generate SQL from a question
# question = "give me the list of employees who were hired after 2020"
# # Step 2: Generate SQL from a question
# sql = query_chain.invoke({"question": question})

# # Step 3: Execute query
# sql_result = db.run(sql)

# # Step 4: Analyze results directly
# analysis_prompt = PromptTemplate.from_template("""
# You are a data analyst. Here is the SQL result:

# {data}

# Answer the user's question: {question}
# """)
# analysis_chain = LLMChain(llm=llm, prompt=analysis_prompt)

# analysis = analysis_chain.run(data=sql_result, question=question)

# print("SQL Query:", sql)
# print("Result:", sql_result)
# print("Analysis:", analysis)


In [41]:
# Initialize the Ollama model

llm = Ollama(model="qwen2.5:7b", base_url="http://192.168.18.8:11434/")
llm.invoke("Hello, Ollama!")
# Test the model with a simple prompt
response = llm.invoke("What is the capital of France?")
print("Model test response:", response)

Model test response: The capital of France is Paris.


In [42]:
import sqlite3

# Create (or connect to) the database
conn = sqlite3.connect("heart_disease.db")
cursor = conn.cursor()

# Drop tables if they exist (for reruns)
cursor.execute("DROP TABLE IF EXISTS Treatments;")
cursor.execute("DROP TABLE IF EXISTS Medical_Records;")
cursor.execute("DROP TABLE IF EXISTS Patients;")

# Create Patients Table
cursor.execute("""
CREATE TABLE Patients (
    patient_id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    gender TEXT,
    bmi REAL,
    smoker INTEGER
);
""")

# Create Medical Records Table
cursor.execute("""
CREATE TABLE Medical_Records (
    record_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    cholesterol INTEGER,
    blood_pressure TEXT,
    diabetes INTEGER,
    heart_disease INTEGER,
    FOREIGN KEY(patient_id) REFERENCES Patients(patient_id)
);
""")

# Create Treatments Table
cursor.execute("""
CREATE TABLE Treatments (
    treatment_id INTEGER PRIMARY KEY,
    patient_id INTEGER,
    medication TEXT,
    surgery TEXT,
    follow_up_months INTEGER,
    FOREIGN KEY(patient_id) REFERENCES Patients(patient_id)
);
""")

# Insert Sample Patients
cursor.executemany("""
INSERT INTO Patients (name, age, gender, bmi, smoker) VALUES (?, ?, ?, ?, ?)
""", [
    ('Ali Khan', 54, 'Male', 28.5, 1),
    ('Sara Ahmed', 47, 'Female', 25.2, 0),
    ('John Doe', 60, 'Male', 30.1, 1),
    ('Maryam Bibi', 35, 'Female', 22.4, 0),
    ('Ahmed Raza', 50, 'Male', 27.8, 1),
])

# Insert Sample Medical Records
cursor.executemany("""
INSERT INTO Medical_Records (patient_id, cholesterol, blood_pressure, diabetes, heart_disease) 
VALUES (?, ?, ?, ?, ?)
""", [
    (1, 240, '145/95', 1, 1),
    (2, 190, '125/80', 0, 0),
    (3, 260, '160/100', 1, 1),
    (4, 180, '110/70', 0, 0),
    (5, 220, '140/90', 0, 1),
])

# Insert Sample Treatments
cursor.executemany("""
INSERT INTO Treatments (patient_id, medication, surgery, follow_up_months) 
VALUES (?, ?, ?, ?)
""", [
    (1, 'Statins', 'Angioplasty', 12),
    (2, 'None', 'None', 6),
    (3, 'Beta Blockers', 'Bypass Surgery', 18),
    (4, 'None', 'None', 6),
    (5, 'ACE Inhibitors', 'Stent Placement', 12),
])

# Commit and close
conn.commit()
conn.close()

print("✅ Database with Patients, Medical_Records, and Treatments tables created successfully!")


✅ Database with Patients, Medical_Records, and Treatments tables created successfully!
